# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates the loading and exploration of the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [mlcroissant](https://github.com/mlcommons/croissant) library. We'll use only entity references by their `@id` fields, following best FAIR and Croissant practices.

### Dataset Source
The dataset source is provided via its Croissant schema URL and is fully described in JSON-LD.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")# Metadata object is used as an object (not dict or list), following mlcroissant's API

## 2. Data Overview
Review available record sets, fields, and their `@id`s for exploration and extraction.

In [ ]:
# List record sets by @id
print("Available record sets and their field @ids:")
record_sets = metadata.record_sets

for rs in record_sets:
    print(f"\nRecord Set: '@id': {rs['@id']}")
    fields = rs.get('fields', [])
    if fields:
        for field in fields:
            print(f"  Field '@id': {field['@id']}, name: {field['name']}, dataType: {field.get('dataType', None)}")
    else:
        print('  (No fields explicitly listed in schema)')

## 3. Data Extraction
Load records for each record set using their `@id`. Fields and columns must be referred to by their `@id` for robust, schema-aligned operations.

In [ ]:
# We'll use the first record set as the main data table (adjust to target specific sets if there are multiple, e.g. for metadata vs table data)
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Record set ids to use: {record_set_ids}")
dataframes = {}

for record_set_id in record_set_ids:
    # Load records as list of dicts
    print(f"\nLoading records from Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  {len(df)} records, columns: {list(df.columns)}")
    else:
        print('  No records found or empty record set.')

# For illustration, pick the main record set for analysis (e.g. a table of proteins). Update this if a specific @id should be used, otherwise, the first one is used.
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    if main_record_set_id in dataframes:
        print(f"\nHead of DataFrame for {main_record_set_id}:")
        display(dataframes[main_record_set_id].head())
    else:
        print(f'No DataFrame loaded for main record set {main_record_set_id}')
else:
    main_record_set_id = None
    print('No record sets found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. This example operates with fields by their `@id`, and demonstrates filtering, normalization, and grouping if applicable.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Example: Select a numeric field by its @id (adjust this if schema shows fields/columns with @id and appropriate dataType)
    # Below we attempt to pick a likely numeric column (common for protein datasets)
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not numeric_candidates:
        # Try to infer numerics from data (sometimes CSV loads numbers as objects)
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
        if threshold is not None:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
            display(filtered_df.head())

            # Normalization
            norm_col = f"{numeric_field_id}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} (z-score):")
            display(filtered_df[[numeric_field_id, norm_col]].head())

            # Group by a categorical field (if any exist)
            group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
            if group_candidates:
                group_field_id = group_candidates[0]
                print(f"Grouping by: {group_field_id}")
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(grouped_df.head())
            else:
                print('No suitable categorical field for grouping found.')
        else:
            print('No suitable numeric data for EDA.')
    else:
        print('No numeric fields found for EDA.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualizing distributions and relationships. Example: histogram of a numeric field and scatter against another numeric field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Try to reselect numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) >= 1:
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_cols[0]].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_cols[0]}")
        plt.xlabel(numeric_cols[0])
        plt.show()
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(5,5))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.title(f"Scatter: {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, manipulate, and visualize a FAIR dataset using the `mlcroissant` library. We:
- Inspected record sets and fields via their `@id` following Croissant best practices.
- Loaded tables as DataFrames, filtered and normalized a numeric field, and grouped by a categorical variable when possible.
- Visualized underlying data distributions with common plots.

For advanced usage, refer to the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) and the [FAIR^2 dataset repo](https://sen.science/doi/10.71728/senscience.vq4a-28xa).